# LDA Topic Discovery — NFL Scouting Reports

**Goal:** Run LDA with 4 topics on preprocessed scouting text and see whether the model
recovers something resembling the four pillars (physical / technique / character / IQ)
without any supervision.

Uses the same preprocessing pipeline as `pillar_scoring_tokens.ipynb`:
phrase stitching → NFL-aware stop words → lemmatization.

In [1]:
import pandas as pd
import numpy as np
import re
from collections import Counter

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from gensim import corpora
from gensim.models import LdaModel
from gensim.models.coherencemodel import CoherenceModel

nltk.download('wordnet',   quiet=True)
nltk.download('omw-1.4',   quiet=True)
nltk.download('stopwords', quiet=True)

# ── Config ─────────────────────────────────────────────────────────────────────
N_TOPICS      = 4     # one per pillar
N_PASSES      = 20    # LDA training passes
RANDOM_STATE  = 42
TOP_N_WORDS   = 20    # words to display per topic

FILTER_SPECIALISTS = True

print('Imports OK')

Imports OK


## Section 1 — Load Data

In [2]:
df = pd.read_csv('../data/processed/draft_enriched_with_contracts.csv')

keep_cols = ['player_name', 'Pos_Group', 'position', 'grade', 'year',
             'made_it_contract', 'overview', 'strengths', 'weaknesses']
df = df[[c for c in keep_cols if c in df.columns]].copy()

for col in ['overview', 'strengths', 'weaknesses']:
    df[col] = df[col].fillna('')

if FILTER_SPECIALISTS:
    n_before = len(df)
    df = df[df['Pos_Group'] != 'SPECIAL'].reset_index(drop=True)
    print(f'Specialists removed: {n_before - len(df)}')

print(f'Players: {len(df)}')

Specialists removed: 223
Players: 7173


## Section 2 — Preprocessing

Identical pipeline to `pillar_scoring_tokens.ipynb`:
phrase stitching → stopword removal → lemmatization.

In [3]:
# ── Curated compound NFL terms ─────────────────────────────────────────────────
_CURATED_RAW = {
    'change of direction':      'change_of_direction',
    'low pad level':            'low_pad_level',
    'run after catch':          'run_after_catch',
    'yards after contact':      'yards_after_contact',
    'yards after catch':        'yards_after_catch',
    'run after contact':        'run_after_contact',
    'off the line':             'off_the_line',
    'off the ball':             'off_the_ball',
    'point of attack':          'point_of_attack',
    'get off the line':         'get_off_the_line',
    'read and react':           'read_and_react',
    'inside the box':           'inside_the_box',
    'stack and shed':           'stack_and_shed',
    'pass rush moves':          'pass_rush_moves',
    'tackles for loss':         'tackles_for_loss',
    'tackle for loss':          'tackle_for_loss',
    'setting the edge':         'setting_edge',
    'set the edge':             'setting_edge',
    'turn the corner':          'turn_the_corner',
    'flatten the arc':          'flatten_arc',
    'low to the ground':        'low_to_ground',
    'beat the punch':           'beat_the_punch',
    'grease the edge':          'grease_edge',
    'block take on':            'block_takeon',
    'block take-on':            'block_takeon',
    'knock back pop':           'knock_back_pop',
    'furious finish':           'furious_finish',
    'furious finishes':         'furious_finish',
    'make plays':               'play_making',
    'makes plays':              'play_making',
    'generates pressure':       'generate_pressure',
    'generate pressure':        'generate_pressure',
    'creates pressure':         'generate_pressure',
    'create pressure':          'generate_pressure',
    'north south':              'north_south',
    'north-south':              'north_south',
    'balanced base':            'balanced_base',
    'wide base':                'wide_base',
    'lower half power':         'lower_half',
    'explosive hips':           'explosive_hips',
    'burst of speed':           'burst_of_speed',
    'arm strength':             'arm_strength',
    'leg drive':                'leg_drive',
    'catch radius':             'catch_radius',
    'catching radius':          'catch_radius',
    'hip tightness':            'hip_tightness',
    'tight hips':               'hip_tightness',
    'high tightness':           'high_tightness',
    'hip bend':                 'hip_bend',
    'bend the edge':            'bend_edge',
    'edge bending':             'bend_edge',
    'lean frame':               'lean_frame',
    'lean build':               'lean_build',
    'thick frame':              'thick_frame',
    'raw speed':                'raw_speed',
    'raw power':                'raw_power',
    'raw athleticism':          'raw_athleticism',
    'long strides':             'long_strides',
    'short strides':            'short_strides',
    'straight line speed':      'straight_line_speed',
    'straight-line speed':      'straight_line_speed',
    'grip strength':            'grip_strength',
    'wrist flick':              'wrist_flick',
    'release point':            'release_point',
    'arm angles':               'arm_angles',
    'arm angle':                'arm_angles',
    'sees pressure':            'blitz_awareness',
    'reads pressure':           'blitz_awareness',
    'tight window':             'tight_windows',
    'tight windows':            'tight_windows',
    'drive throw':              'drive_throw',
    'drive throws':             'drive_throw',
    'touch throw':              'touch_pass',
    'cannon arm':               'cannon',
    'quick release':            'quick_delivery',
    'under pressure':           'pressure',
    'pocket passer':            'pocket_passer',
    'lower half':               'lower_half',
    'lower body':               'lower_body',
    'upper body':               'upper_body',
    'playing speed':            'playing_speed',
    'change of pace':           'change_of_pace',
    'playing weight':           'playing_weight',
    'body weight':              'body_weight',
    'arm length':               'arm_length',
    'violent club':             'violent_club',
    'club move':                'club_move',
    'spin move':                'spin_move',
    'swim move':                'swim_move',
    'speed rush':               'speed_rush',
    'counter move':             'counter_move',
    'rush move':                'rush_moves',
    'rush moves':               'rush_moves',
    'bull rush':                'bull_rush',
    'bull rusher':              'bull_rusher',
    'bull rushing':             'bull_rush',
    'push pocket':              'push_pocket',
    'collapse pocket':          'collapse_pocket',
    'attack angle':             'attack_angle',
    'rush plan':                'rush_plan',
    'run support':              'run_support',
    'run stopper':              'run_stopper',
    'setting edge':             'setting_edge',
    'run fits':                 'run_fits',
    'run fit':                  'run_fits',
    'run defense':              'run_defense',
    'pursuit angle':            'pursuit_angle',
    'wrap up':                  'wrap_up',
    'sure tackler':             'sure_tackler',
    'secure tackler':           'sure_tackler',
    'downhill trigger':         'downhill_trigger',
    'pass sets':                'pass_sets',
    'arm extension':            'arm_extension',
    'drive blocking':           'drive_blocking',
    'reach blocking':           'reach_blocking',
    'zone blocker':             'zone_blocker',
    'gap blocker':              'gap_blocker',
    'road grader':              'road_grader',
    'active hands':             'active_hands',
    'hand usage':               'hand_usage',
    'hip sink':                 'hip_sink',
    'pocket presence':          'pocket_presence',
    'edge setter':              'edge_setter',
    'press technique':          'press_technique',
    'off coverage':             'off_coverage',
    'trail technique':          'trail_technique',
    'man to man':               'man_coverage',
    'zone scheme':              'zone_coverage',
    'zone drops':               'zone_drops',
    're route':                 're_route',
    're-route':                 're_route',
    'break on ball':            'break_on_ball',
    'high point':               'high_point',
    'contested catch':          'contested_catch',
    'contested catches':        'contested_catch',
    'post up':                  'post_up',
    'frame up':                 'frame_up',
    'separation quickness':     'separation_quickness',
    'football intelligence':    'football_intelligence',
    'mental toughness':         'mental_toughness',
    'high effort':              'high_effort',
    'off field':                'off_field',
    'practice habits':          'practice_habits',
    'film room':                'film_room',
    'football character':       'football_character',
    'blitz awareness':          'blitz_awareness',
    'pocket awareness':         'pocket_awareness',
    'eye level':                'eye_level',
    'ultra competitive':        'ultra_competitive',
    'pass rush':                'pass_rush',
    'pass rusher':              'pass_rusher',
    'pass protection':          'pass_protection',
    'pass coverage':            'pass_coverage',
    'pad level':                'pad_level',
    'press coverage':           'press_coverage',
    'man coverage':             'man_coverage',
    'zone coverage':            'zone_coverage',
    'zone coverages':           'zone_coverages',
    'ball skills':              'ball_skills',
    'ball hawk':                'ball_hawk',
    'ball hawking':             'ball_hawking',
    'ball carrier':             'ball_carrier',
    'ball carriers':            'ball_carriers',
    'body control':             'body_control',
    'contact balance':          'contact_balance',
    'closing speed':            'closing_speed',
    'lateral quickness':        'lateral_quickness',
    'quick twitch':             'quick_twitch',
    'high motor':               'high_motor',
    'first step':               'first_step',
    'get off':                  'get_off',
    'hand fighting':            'hand_fighting',
    'hand strength':            'hand_strength',
    'hand placement':           'hand_placement',
    'strong hands':             'strong_hands',
    'soft hands':               'soft_hands',
    'heavy hands':              'heavy_hands',
    'block shedding':           'block_shedding',
    'anchor strength':          'anchor_strength',
    'route running':            'route_running',
    'run blocking':             'run_blocking',
    'open field':               'open_field',
    'red zone':                 'red_zone',
    'second level':             'second_level',
    'hip flexibility':          'hip_flexibility',
    'short area':               'short_area',
    'three down':               'three_down',
    'top end':                  'top_end',
    'two gap':                  'two_gap',
    'one gap':                  'one_gap',
    'two gapping':              'two_gapping',
    'one gapping':              'one_gapping',
    'snap count':               'snap_count',
    'football iq':              'football_iq',
    'film study':               'film_study',
    'work ethic':               'work_ethic',
    'locker room':              'locker_room',
    'decision making':          'decision_making',
    'play making':              'play_making',
    'field vision':             'field_vision',
    'play recognition':         'play_recognition',
    'pre snap':                 'pre_snap',
    'post snap':                'post_snap',
    'game ready':               'game_ready',
    'hard worker':              'hard_worker',
    'arm_length':               'arm_length',
    'functional strength':      'functional_strength',
    'tackling form':            'tackling_form',
    'processing speed':         'processing_speed',
    'eye discipline':           'eye_discipline',
    'spatial awareness':        'spatial_awareness',
    'route progressions':       'route_progressions',
}

CURATED_PHRASE_MAP = dict(
    sorted(_CURATED_RAW.items(), key=lambda x: len(x[0]), reverse=True)
)
print(f'Curated phrases: {len(CURATED_PHRASE_MAP)}')

# ── NFL-aware stop words ────────────────────────────────────────────────────────
KEEP_WORDS = {
    'high', 'low', 'heavy', 'light', 'deep', 'short', 'long', 'wide',
    'hard', 'soft', 'strong', 'quick', 'good', 'great',
    'up', 'down', 'off', 'out', 'over', 'through', 'above', 'below',
    'hand', 'hands', 'back', 'field', 'ball'
}

CUSTOM_STOPS = {
    'prospect', 'player', 'players', 'show', 'shows', 'need', 'needs',
    'also', 'often', 'must', 'well', 'still', 'use', 'get',
    'make', 'look', 'help', 'time', 'year', 'team', 'game',
    'continue', 'develop', 'development', 'nfl', 'draft', 'college',
    'type', 'project', 'potential', 'upside',
    'starter', 'backup', 'senior', 'junior', 'cb', 'rb', 'wr', 'qb'
}

_base = set(stopwords.words('english'))
NFL_STOPWORDS = (_base - KEEP_WORDS) | CUSTOM_STOPS
print(f'Stop words: {len(NFL_STOPWORDS)}')

Curated phrases: 212
Stop words: 229


In [4]:
lemmatizer = WordNetLemmatizer()

def nfl_preprocess(text: str, phrase_map: dict = CURATED_PHRASE_MAP) -> str:
    """NFL-aware preprocessing: normalize → stitch → clean → filter → lemmatize."""
    if not isinstance(text, str) or not text.strip():
        return ''
    text = text.lower()
    text = re.sub(r'[-\u2013\u2014]', ' ', text)
    for phrase, token in phrase_map.items():
        text = text.replace(phrase, token)
    text = re.sub(r'[^a-z_\s]', ' ', text)
    tokens = text.split()
    tokens = [t for t in tokens if '_' in t or t not in NFL_STOPWORDS]
    tokens = [t if '_' in t else lemmatizer.lemmatize(t) for t in tokens]
    tokens = [t for t in tokens if len(t) > 1]
    return ' '.join(tokens)

df['all_text']   = (
    df['overview'].apply(nfl_preprocess) + ' ' +
    df['strengths'].apply(nfl_preprocess) + ' ' +
    df['weaknesses'].apply(nfl_preprocess)
).str.strip()
df['all_tokens'] = df['all_text'].apply(str.split)

# Drop players with empty reports
n_before = len(df)
df = df[df['all_tokens'].apply(len) > 0].reset_index(drop=True)
print(f'Preprocessing complete.  Dropped {n_before - len(df)} empty reports.')
print(f'Players: {len(df)}')
lengths = df['all_tokens'].apply(len)
print(f'Tokens per player — median: {lengths.median():.0f},  min: {lengths.min()},  max: {lengths.max()}')

Preprocessing complete.  Dropped 653 empty reports.
Players: 6520
Tokens per player — median: 125,  min: 26,  max: 396


## Section 3 — LDA Config

In [5]:
N_TOPICS       = 4     # topics per group
N_PASSES       = 20    # LDA training passes
RANDOM_STATE   = 42
TOP_N_WORDS    = 20    # words shown per topic
MIN_GROUP_SIZE = 50    # skip groups with fewer players

# Spotlight: run these two first (col, value)
SPOTLIGHT_GROUPS = [('Pos_Group', 'QB'), ('Pos_Group', 'DB')]

# Then loop over all unique values of this column
LOOP_COL  = 'Group'
LOOP_SKIP = {'SPECIAL'}

print('LDA config set.')

LDA config set.


## Section 4 — LDA Helper

`run_lda_for_subset(label, sub_df)` builds a corpus, trains LDA, prints topics +
coherence, and stores results in module-level dicts for downstream use.

In [6]:
# Global result stores — keyed by label
lda_models   = {}
dictionaries = {}
lda_corpora  = {}
group_dfs    = {}


def run_lda_for_subset(label: str, sub_df: pd.DataFrame) -> None:
    """Train LDA on a subset of players, print topics + coherence, store results."""
    token_lists = sub_df['all_tokens'].tolist()
    n = len(token_lists)

    if n < MIN_GROUP_SIZE:
        print(f'Skipping {label} (n={n} < {MIN_GROUP_SIZE})')
        return

    # ── Build corpus ──────────────────────────────────────────────────────────
    dictionary = corpora.Dictionary(token_lists)
    dictionary.filter_extremes(no_below=5, no_above=0.80)
    dictionary.compactify()
    corpus = [dictionary.doc2bow(tokens) for tokens in token_lists]

    if len(dictionary) < N_TOPICS * 3:
        print(f'Skipping {label} — vocabulary too small ({len(dictionary)} terms)')
        return

    # ── Train LDA ─────────────────────────────────────────────────────────────
    lda = LdaModel(
        corpus       = corpus,
        id2word      = dictionary,
        num_topics   = N_TOPICS,
        passes       = N_PASSES,
        random_state = RANDOM_STATE,
        alpha        = 'auto',
        eta          = 'auto',
        eval_every   = None,
    )

    # ── Store ─────────────────────────────────────────────────────────────────
    lda_models[label]   = lda
    dictionaries[label] = dictionary
    lda_corpora[label]  = corpus
    group_dfs[label]    = sub_df.reset_index(drop=True)

    # ── Print topics ──────────────────────────────────────────────────────────
    print(f'\n{"=" * 62}')
    print(f'  {label}  (n={n}, vocab={len(dictionary)})')
    print(f'{"=" * 62}')
    for topic_id in range(N_TOPICS):
        top_words = lda.show_topic(topic_id, topn=TOP_N_WORDS)
        words_str = ',  '.join(f'{w} ({p:.3f})' for w, p in top_words)
        print(f'  Topic {topic_id}: {words_str}')

    # ── Coherence ─────────────────────────────────────────────────────────────
    coherence_model = CoherenceModel(
        model=lda, texts=token_lists, dictionary=dictionary, coherence='c_v'
    )
    print(f'\n  Coherence (C_v): {coherence_model.get_coherence():.4f}')


print('Helper defined.')

Helper defined.


## Section 5 — Spotlight: QB and DB

Run first so you can immediately inspect the two most interpretable groups
before the full loop output arrives.

In [7]:
for col, val in SPOTLIGHT_GROUPS:
    sub = df[df[col] == val].copy()
    run_lda_for_subset(f'{val}', sub)


  QB  (n=300, vocab=1265)
  Topic 0: field (0.012),  up (0.009),  off (0.009),  arm (0.009),  play (0.009),  good (0.008),  out (0.008),  accuracy (0.008),  offense (0.007),  pro (0.007),  move (0.007),  foot (0.006),  passing (0.006),  receiver (0.006),  could (0.006),  down (0.006),  deep (0.006),  throwing (0.005),  read (0.005),  mobility (0.005)
  Topic 1: good (0.017),  quarterback (0.013),  accuracy (0.011),  down (0.011),  arm_strength (0.011),  level (0.011),  deep (0.009),  enough (0.009),  ability (0.009),  foot (0.008),  next (0.008),  out (0.008),  many (0.007),  read (0.007),  up (0.007),  arm (0.007),  lot (0.007),  field (0.007),  progression (0.007),  offense (0.006)
  Topic 2: good (0.011),  quarterback (0.011),  play (0.010),  out (0.009),  could (0.008),  set (0.008),  foot (0.008),  arm_strength (0.008),  field (0.008),  short (0.007),  up (0.007),  take (0.007),  athletic (0.007),  deep (0.007),  accuracy (0.007),  level (0.007),  size (0.007),  throwing (0.006),

## Section 6 — All Group Values

In [8]:
for group_val in sorted(df[LOOP_COL].dropna().unique()):
    if group_val in LOOP_SKIP:
        continue
    sub = df[df[LOOP_COL] == group_val].copy()
    run_lda_for_subset(group_val, sub)

KeyError: 'Group'

## Section 7 — Spot Check: Top Players per Topic

For each group × topic, show the 5 players with the highest topic loading.
Use these to read their scouting reports and manually label what each topic represents.

In [ ]:
SPOT_N = 5  # players to show per topic

for label, lda in lda_models.items():
    sub_df  = group_dfs[label]
    corpus_ = lda_corpora[label]

    def _get_dist(bow):
        d = dict(lda.get_document_topics(bow, minimum_probability=0.0))
        return [d.get(i, 0.0) for i in range(N_TOPICS)]

    topic_mat = np.array([_get_dist(bow) for bow in corpus_])

    print(f'\n{"=" * 62}')
    print(f'  {label} — top {SPOT_N} players per topic')
    print(f'{"=" * 62}')

    for t in range(N_TOPICS):
        top_idx = topic_mat[:, t].argsort()[::-1][:SPOT_N]
        top_words = ', '.join(w for w, _ in lda.show_topic(t, topn=5))
        print(f'\n  Topic {t}  [{top_words}]')
        for i in top_idx:
            row = sub_df.iloc[i]
            pct = topic_mat[i, t] * 100
            pos = row.get('position', '?')
            print(f'    {row["player_name"]:28s}  {pos:6s}  {pct:.1f}%')

## Section 8 — Compare to Pillar Scores (Optional)

If `bin_scores_v2.csv` exists, compute per-group correlation between LDA topic
probabilities and the keyword-based physical / technique / character pillar scores.
A strong correlation suggests LDA is recovering similar structure.

In [ ]:
import os

scores_path = '../data/processed/bin_scores_v2.csv'
if not os.path.exists(scores_path):
    print(f'Skipping — {scores_path} not found. Run pillar_scoring_tokens.ipynb first.')
else:
    pillar = pd.read_csv(scores_path)[
        ['player_name', 'year', 'physical_pct', 'technique_pct', 'character_pct']
    ]

    pillar_cols = ['physical_pct', 'technique_pct', 'character_pct']
    topic_cols  = [f'topic_{i}_pct' for i in range(N_TOPICS)]

    for label, lda in lda_models.items():
        sub_df  = group_dfs[label]
        corpus_ = lda_corpora[label]

        def _get_dist(bow):
            d = dict(lda.get_document_topics(bow, minimum_probability=0.0))
            return [d.get(i, 0.0) for i in range(N_TOPICS)]

        topic_mat = np.array([_get_dist(bow) for bow in corpus_])
        sub = sub_df.copy()
        for i in range(N_TOPICS):
            sub[f'topic_{i}_pct'] = topic_mat[:, i] * 100

        merged = sub.merge(pillar, on=['player_name', 'year'], how='inner')
        if len(merged) < 20:
            print(f'{label}: too few matches to correlate (n={len(merged)})')
            continue

        corr = merged[topic_cols + pillar_cols].corr().loc[topic_cols, pillar_cols]
        print(f'\n{label}  (n={len(merged)}) — topic vs pillar correlation:')
        print(corr.round(3).to_string())

## Section 9 — Top Words & Bigrams by Position

For each position group: show the most frequent **unigrams** and **bigrams** after
preprocessing. Use this to spot high-frequency scouting vocabulary that may be missing
from `ANCHOR_SEEDS` or `CURATED_PHRASE_MAP` in `pillar_scoring_tokens.ipynb`.

Bigrams are built from consecutive preprocessed tokens — compound phrases
(e.g. `pass_rush`, `route_running`) already appear as single tokens via
`CURATED_PHRASE_MAP`, so bigrams here reveal *new* multi-word patterns worth stitching.

In [ ]:
from collections import Counter

# ── Config ─────────────────────────────────────────────────────────────────────
TOP_N_WORDS   = 40   # unigrams to show per position
TOP_N_BIGRAMS = 30   # bigrams to show per position
MIN_UNIGRAM_FREQ = 10  # ignore tokens appearing fewer than N times in this group
MIN_BIGRAM_FREQ  = 5

# Optional: load covered keyword set from tokens notebook output for comparison.
# Set to None to skip coverage tagging.
import os
_lex_path = '../data/processed/bin_lexicons_v2.csv'
if os.path.exists(_lex_path):
    _lex_df  = pd.read_csv(_lex_path)
    COVERED  = set(_lex_df['term'].tolist())
    print(f'Loaded {len(COVERED)} covered terms from bin_lexicons_v2.csv')
else:
    COVERED = set()
    print('bin_lexicons_v2.csv not found — coverage tagging disabled')

# ── Per-position analysis ─────────────────────────────────────────────────────
for pos_group in sorted(df['Pos_Group'].dropna().astype(str).unique()):
    sub = df[df['Pos_Group'] == pos_group]
    if len(sub) < 20:
        continue

    # ── Unigrams ──────────────────────────────────────────────────────────────
    uni_counter = Counter()
    for toks in sub['all_tokens']:
        if isinstance(toks, list):
            uni_counter.update(toks)

    top_uni = [(t, f) for t, f in uni_counter.most_common()
               if f >= MIN_UNIGRAM_FREQ][:TOP_N_WORDS]

    # ── Bigrams ───────────────────────────────────────────────────────────────
    bi_counter = Counter()
    for toks in sub['all_tokens']:
        if isinstance(toks, list) and len(toks) >= 2:
            # Only pair tokens that are both single words (no existing compounds)
            # so we surface genuinely new multi-word patterns
            bi_counter.update(
                f'{toks[i]} {toks[i+1]}'
                for i in range(len(toks) - 1)
                if '_' not in toks[i] and '_' not in toks[i+1]
            )

    top_bi = [(bg, f) for bg, f in bi_counter.most_common()
              if f >= MIN_BIGRAM_FREQ][:TOP_N_BIGRAMS]

    # ── Display ───────────────────────────────────────────────────────────────
    print(f'\n{"=" * 68}')
    print(f'  {pos_group}  (n={len(sub)})')
    print(f'{"=" * 68}')

    print(f'\n  TOP {TOP_N_WORDS} UNIGRAMS')
    print(f'  {"RANK":<5} {"TOKEN":<30} {"FREQ":>6}  {"COVERED?"}')
    print(f'  {"-"*55}')
    for rank, (tok, freq) in enumerate(top_uni, 1):
        tag = '✓' if tok in COVERED else ' '
        print(f'  {rank:<5} {tok:<30} {freq:>6}  {tag}')

    print(f'\n  TOP {TOP_N_BIGRAMS} BIGRAMS  (new multi-word patterns)')
    print(f'  {"RANK":<5} {"BIGRAM":<35} {"FREQ":>6}')
    print(f'  {"-"*50}')
    for rank, (bg, freq) in enumerate(top_bi, 1):
        print(f'  {rank:<5} {bg:<35} {freq:>6}')